# 06 – Modelling Cohort & Label Construction

Narrows the full Angus cohort to the modelling cohort and defines the outcome label:
keep sepsis-positive stays, require a complete 24-hour ICU window, exclude deaths within
the first 24h (label leakage), and label post-24-hour in-hospital mortality.

**Run after notebook 02** — uses final_cohort_angus.csv.

**Produces:** modelling_cohort_sepsis_mortality.csv (used by notebooks 01, 03, 04, 05, 07, 08, 10, 11).

MIMIC-III data not included (PhysioNet DUA); see README.

In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import pandas as pd
import numpy as np


In [ ]:
cohort = pd.read_csv(data_path("final_cohort_angus.csv"), low_memory=False)
print("Full Angus cohort:", cohort.shape)

Full Angus cohort: (33560, 24)


## 1. Parse datetimes

In [ ]:
for col in ["INTIME", "OUTTIME", "ADMITTIME", "DISCHTIME", "DEATHTIME"]:
    cohort[col] = pd.to_datetime(cohort[col], errors="coerce")

# 24-hour cutoff from ICU admission
cohort["CUTOFF_24H"] = cohort["INTIME"] + pd.Timedelta(hours=24)

## 2. Step 1 — keep sepsis-positive stays only

In [ ]:
sepsis = cohort[cohort["Sepsis_Angus"] == 1].copy()
print("Sepsis-positive ICU stays:", len(sepsis))

Sepsis-positive ICU stays: 10124


## 3. Step 2 — require a complete 24-hour ICU observation window
Keep only stays where the patient was still in the ICU at the 24-hour mark
(`OUTTIME - INTIME >= 24h`). Stays discharged/transferred out of ICU within 24h
do not have a full first-24h signal window.

In [ ]:
icu_duration = (sepsis["OUTTIME"] - sepsis["INTIME"]).dt.total_seconds() / 3600.0
sepsis["ICU_HOURS"] = icu_duration

has_full_window = sepsis["ICU_HOURS"] >= 24
print("Stays with >= 24h ICU window:", has_full_window.sum())
print("Excluded (ICU < 24h):", (~has_full_window).sum())

sepsis_24h = sepsis[has_full_window].copy()

Stays with >= 24h ICU window: 10124
Excluded (ICU < 24h): 0


## 4. Step 3 — exclude deaths within the first 24 hours
If a patient died at or before INTIME + 24h, the outcome falls inside the
observation window and cannot be predicted from it (label leakage).

In [ ]:
died_within_24h = (
    sepsis_24h["DEATHTIME"].notna()
    & (sepsis_24h["DEATHTIME"] <= sepsis_24h["CUTOFF_24H"])
)
print("Excluded (died within first 24h):", died_within_24h.sum())

modelling = sepsis_24h[~died_within_24h].copy()
print("Modelling cohort size:", len(modelling))

Excluded (died within first 24h): 56
Modelling cohort size: 10068


## 5. Step 4 — define the label: post-24-hour in-hospital mortality
`mortality_after_24h = 1` if the admission ended in in-hospital death
(HOSPITAL_EXPIRE_FLAG == 1) — by construction any such death is now after 24h,
since deaths within 24h were excluded above.

In [ ]:
modelling["mortality_after_24h"] = (
    modelling["HOSPITAL_EXPIRE_FLAG"].fillna(0).astype(int)
)

# sanity: every positive death should be after the 24h cutoff (or DEATHTIME missing)
late_or_missing = (
    modelling["DEATHTIME"].isna()
    | (modelling["DEATHTIME"] > modelling["CUTOFF_24H"])
)
print("All positives after 24h cutoff:", bool(late_or_missing.all()))

All positives after 24h cutoff: True


## 6. Final modelling cohort summary

In [ ]:
n = len(modelling)
n_pos = int(modelling["mortality_after_24h"].sum())
n_neg = n - n_pos

print("=== Final modelling cohort ===")
print("Total sepsis ICU stays:", n)
print("Died in hospital after 24h (label=1):", n_pos)
print("Survived (label=0):", n_neg)
print("Mortality rate: {:.1%}".format(n_pos / n))

=== Final modelling cohort ===
Total sepsis ICU stays: 10068
Died in hospital after 24h (label=1): 2082
Survived (label=0): 7986
Mortality rate: 20.7%


In [ ]:
# Funnel recap
print("Funnel:")
print("  Full Angus cohort:            ", len(cohort))
print("  -> Sepsis-positive:           ", len(sepsis))
print("  -> with >=24h ICU window:     ", len(sepsis_24h))
print("  -> excl. death within 24h:    ", len(modelling))

Funnel:
  Full Angus cohort:             33560
  -> Sepsis-positive:            10124
  -> with >=24h ICU window:      10124
  -> excl. death within 24h:     10068


## 7. Save the modelling cohort

In [ ]:
cols_to_keep = [
    "SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "INTIME", "OUTTIME",
    "AGE", "GENDER", "LOS", "ICU_HOURS",
    "Sepsis_Angus", "AKI", "HOSPITAL_EXPIRE_FLAG",
    "DEATHTIME", "mortality_after_24h"
]
modelling_out = modelling[cols_to_keep].copy()
modelling_out.to_csv(data_path("modelling_cohort_sepsis_mortality.csv"), index=False)
print("Saved:", data_path("modelling_cohort_sepsis_mortality.csv"))
print("Shape:", modelling_out.shape)

In [ ]:
import pandas as pd
m = pd.read_csv(data_path("modelling_cohort_sepsis_mortality.csv"))
print("sepsis age median:", m["AGE"].median())
print("sepsis male %:", (m["GENDER"]=="M").mean()*100)
print("sepsis LOS median:", m["LOS"].median())

sepsis age median: 68.0
sepsis male %: 53.07906237584425
sepsis LOS median: 4.6687
